In [4]:
import os
from langchain_openai import ChatOpenAI
from IPython.display import display, Markdown

In [1]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [27]:
llm = ChatOpenAI(model="openai/gpt-oss-20b", base_url="https://api.groq.com/openai/v1", api_key=os.getenv("GROQ_API_KEY"))
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

Why do LLM‑engineering students never play hide‑and‑seek?  
Because every time they try to hide, the model just *masks* them and says, “I see you!”

In [28]:
print(response.usage_metadata)

{'input_tokens': 88, 'output_tokens': 1315, 'total_tokens': 1403, 'input_token_details': {}, 'output_token_details': {'reasoning': 1266}}


In [24]:
from litellm import completion

response = completion(
    model="groq/openai/gpt-oss-20b",
    messages=tell_a_joke,
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the LLM‑engineering student bring a transformer to the cafeteria?  
Because they heard the best way to “attend” their lunch was to make sure they never got lost in the attention‑matrix!

In [25]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 88
Output tokens: 1990
Total tokens: 2078
Total cost: 0.0604 cents


LiteLLM to illustrate a Pro-feature: prompt caching

In [29]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
  hamlet = f.read()


loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [49]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [50]:
response = completion(
    model="groq/openai/gpt-oss-20b",
    messages=question,
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
display(Markdown(response.choices[0].message.content))

In the play, when Laertes cries out “Where is my father?” the answer is given immediately: **“He is dead.”** The reply is usually spoken by King Claudius (or another character noting that Polonius has been killed).

In [51]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 89
Output tokens: 1903
Total tokens: 1992
Total cost: 0.0578 cents


In [52]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet[loc:loc+100]

In [55]:
response = completion(
    model="groq/openai/gpt-oss-20b",
    messages=question,
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
display(Markdown(response.choices[0].message.content))

In the original text of *Hamlet*, the exchange occurs in **Act 4, Scene 5** after the death of Polonius.  Laertes, upset over his father’s demise, asks:

> **Laertes:** *Where is my father?*  

The king replies simply:

> **King:** *He is dead.*

(Afterward the queen remarks “But not by him!” and the king continues, but the immediate reply to Laertes’s question is “He is dead.”)

In [61]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet

In [75]:
response = completion(model="gemini/gemini-3.1-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

Based on the text of *Hamlet* provided, here is the exchange you are asking about:

> **Laer.** Where is my father?
> **King.** Dead.
> **Queen.** But not by him!
> **King.** Let him demand his fill.

In [76]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached Tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53307
Output tokens: 58
Cached Tokens: None
Total cost: 1.3414 cents


In [77]:
response = completion(model="gemini/gemini-3.1-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

Based on the text of *Hamlet*, the scene you are referring to occurs in **Act IV, Scene V**. When Laertes demands to know the whereabouts of his father, the exchange is as follows:

> **Laer.** Where is my father?
> **King.** Dead.
> **Queen.** But not by him!
> **King.** Let him demand his fill.

In [78]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached Tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 53307
Output tokens: 80
Cached Tokens: 49124
Total cost: 0.2394 cents
